# 02 — Index House → checklist des PTR

**Rôle :** télécharger les index XML annuels officiels de la House (2013-2026), les parser, filtrer les PTR (`FilingType = P`), et produire la checklist `DocID → URL` des PDF à récupérer.

**Source :** `disclosures-clerk.house.gov` — HTTPS simple, **pas d'Akamai, pas de gate**. *Aucune transformation des transactions ici* : on prépare seulement la liste.

**Cellule 1 — Imports & chemins.** Notebook autonome.

In [1]:
import requests, zipfile, time
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() or (p / ".git").exists():
            return p
    return start

PROJECT_ROOT    = find_project_root(Path.cwd())
RAW_HOUSE_INDEX = PROJECT_ROOT / "data" / "raw" / "house" / "index"; RAW_HOUSE_INDEX.mkdir(parents=True, exist_ok=True)
PROC_HOUSE      = PROJECT_ROOT / "data" / "processed" / "house"; PROC_HOUSE.mkdir(parents=True, exist_ok=True)
REPORTS         = PROJECT_ROOT / "reports"; REPORTS.mkdir(parents=True, exist_ok=True)
HOUSE_YEARS     = range(2013, 2027)
REQUEST_PAUSE_S = 0.6
USER_AGENT      = "Ramify-QIS research (contact: <ton-email>)"
print("Années House :", HOUSE_YEARS.start, "->", HOUSE_YEARS.stop - 1)

Années House : 2013 -> 2026


**Cellule 2 — Les deux URL officielles.** Le ZIP annuel d'index, et le PDF d'un PTR à partir de son `DocID`.

In [2]:
HOUSE_ZIP = "https://disclosures-clerk.house.gov/public_disc/financial-pdfs/{year}FD.zip"
HOUSE_PDF = "https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/{year}/{doc_id}.pdf"

def zip_url(year):         return HOUSE_ZIP.format(year=int(year))
def pdf_url(year, doc_id): return HOUSE_PDF.format(year=int(year), doc_id=str(doc_id).strip())

**Cellule 3 — Téléchargement d'un ZIP annuel.** Idempotent (saute si déjà présent), écriture atomique.

In [3]:
def download_zip(year: int, force: bool = False) -> Path:
    path = RAW_HOUSE_INDEX / f"{year}FD.zip"
    if path.exists() and path.stat().st_size > 0 and not force:
        return path
    r = requests.get(zip_url(year), headers={"User-Agent": USER_AGENT}, timeout=60)
    r.raise_for_status()
    tmp = path.with_suffix(".zip.tmp")
    tmp.write_bytes(r.content)
    tmp.replace(path)
    return path

**Cellule 4 — Extraction du XML.** Chaque ZIP contient un fichier `<année>FD.xml`.

In [4]:
def extract_xml(zip_path: Path, year: int) -> Path:
    target = RAW_HOUSE_INDEX / f"{year}FD.xml"
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        xmls = [n for n in names if n.lower().endswith(".xml")]
        chosen = f"{year}FD.xml" if f"{year}FD.xml" in names else xmls[0]
        target.write_bytes(zf.read(chosen))
    return target

**Cellule 5 — Parsing d'un XML annuel.** Un nœud `<Member>` = un dépôt. On extrait type, date, DocID, nom, état/district.

In [5]:
def _txt(node, tag):
    el = node.find(tag)
    return (el.text or "").strip() if el is not None else ""

def parse_year_xml(xml_path: Path, year: int) -> pd.DataFrame:
    root = ET.parse(xml_path).getroot()
    rows = []
    for m in root.findall(".//Member"):
        rows.append({
            "year":        int(year),
            "filing_type": _txt(m, "FilingType"),
            "filing_date": _txt(m, "FilingDate"),   # = disclosure_date côté House
            "doc_id":      _txt(m, "DocID"),
            "last_name":   _txt(m, "Last"),
            "first_name":  _txt(m, "First"),
            "state_dst":   _txt(m, "StateDst"),
        })
    return pd.DataFrame(rows)

**Cellule 6 — Boucle sur toutes les années.** On télécharge, extrait et parse 2013→2026 → index complet des filings.

In [6]:
frames = []
for year in HOUSE_YEARS:
    zp = download_zip(year)
    xp = extract_xml(zp, year)
    frames.append(parse_year_xml(xp, year))
    print(f"{year} : ok")
    time.sleep(REQUEST_PAUSE_S)

filings = pd.concat(frames, ignore_index=True)
print("Total filings :", len(filings))

2013 : ok


2014 : ok


2015 : ok


2016 : ok


2017 : ok


...

2023 : ok


2024 : ok


2025 : ok


2026 : ok


Total filings : 37451


**Cellule 7 — Contrôle des FilingType.** Distribution des codes. `P` = Periodic Transaction Report (PTR) ; `C`=Candidate, `A`=Annual, `X`=Extension, `T`=Termination. On garde un œil sur d'éventuels amendements hors `P`.

In [7]:
filings["filing_type"].value_counts()

filing_type
C    10089
P     8248
X     6162
O     5838
A     2742
D     2285
W     1104
T      468
H      395
G       55
E       40
B       24
R        1
Name: count, dtype: int64

**Cellule 8 — Filtre PTR + checklist.** On garde `FilingType = P`, on construit l'URL PDF, et on sauvegarde l'index PTR (global + par année).

In [8]:
ptr = filings[filings["filing_type"].str.upper() == "P"].copy().reset_index(drop=True)
ptr["pdf_url"] = [pdf_url(y, d) for y, d in zip(ptr["year"], ptr["doc_id"])]

filings.to_csv(PROC_HOUSE / "house_filings_index.csv", index=False)
ptr.to_csv(PROC_HOUSE / "house_ptr_index.csv", index=False)
for y, g in ptr.groupby("year"):
    g.to_csv(PROC_HOUSE / f"ptr_index_{int(y)}.csv", index=False)
print("PTR identifiés :", len(ptr))

PTR identifiés : 8248


**Cellule 9 — Audit de l'index.** PTR par an, DocID manquants, doublons — écrit dans un rapport Markdown.

In [9]:
by_year       = ptr.groupby("year").size()
missing_docid = int((ptr["doc_id"].str.strip() == "").sum())
dups          = int(filings.duplicated(["year", "doc_id"]).sum())

lines = ["# Audit de l'index House", "",
         f"Généré : {datetime.now(timezone.utc).isoformat(timespec='seconds')}", "",
         f"- Filings totaux : {len(filings)}",
         f"- PTR (FilingType=P) : {len(ptr)}",
         f"- DocID manquants : {missing_docid}",
         f"- Doublons year+doc_id : {dups}", "",
         "## PTR par an"]
lines += [f"- {int(y)} : {int(n)}" for y, n in by_year.items()]
(REPORTS / "house_index_audit.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

# Audit de l'index House

Généré : 2026-06-23T21:56:44+00:00

- Filings totaux : 37451
- PTR (FilingType=P) : 8248
- DocID manquants : 0
- Doublons year+doc_id : 18

## PTR par an
- 2013 : 8
- 2014 : 708
- 2015 : 728
- 2016 : 765
- 2017 : 801
- 2018 : 830
- 2019 : 683
- 2020 : 733
- 2021 : 680
- 2022 : 624
- 2023 : 460
- 2024 : 451
- 2025 : 515
- 2026 : 262


Index House prêt ✅ — la checklist `house_ptr_index.csv` alimente **`03_house_download`**.